<a href="https://colab.research.google.com/github/KartikeyaGITRepo/Hands-on-LLM/blob/main/Hands_on_LLM_Chapter_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Chapter 3**

In [6]:
!pip install transformers

In [1]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

In [2]:
print(torch.__version__)

2.11.0+cu130


In [3]:

# Load model and tokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map = "cuda",
    torch_dtype = "auto",
    # trust_remote_code = True,
)


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [8]:
generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    max_new_tokens = 100,
    do_sample = False,
)

In [9]:
prompt = "write an email apologizing to Anamika for the tragic gardenung mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

The email should be formal, express sincere regret, and offer to make amends. Include a suggestion for a future gardening session to help Anamika recover from the loss.

Anamika,

I hope this message finds you in good spirits despite the recent unfortunate events in your garden. I am writing to you with a heavy heart, as I am deeply sorry for the mishap that occurred with your beloved gardening project.

As


In [10]:
# print the sequence of tokenization, transformers, LM head
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [15]:
# token picking or decoding processs

prompt = "The capital of france is"

# tokenize the input prompt and move the tensor to the same device as the model (cuda)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Get the output of model before model head
model_output = model.model(input_ids)

lm_head_output = model.lm_head(model_output[0])

# Get the last layer's hidden states
hidden_states = model_output.last_hidden_state

# Get the logits directly by passing hidden states through the lm_head layer
logits = model.lm_head(hidden_states)

print("Logits shape:", logits.shape)

Logits shape: torch.Size([1, 6, 32064])


In [16]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'par'